<a href="https://colab.research.google.com/github/hiten4/Week8/blob/main/Week_8_Thursday.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# W8 Thursday Assignment
RNN + LSTM + Sequential Data

In [1]:
!pip install pandas numpy matplotlib scikit-learn tensorflow -q

In [2]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense


In [6]:

df = pd.read_csv('Day 47 stock_prices.csv')
df = df.sort_values('date')
df.head()


,date,ticker,open,high,low,close,volume,returns_pct
0,2022-01-03,RELIANCE,2476.84,2496.70,2422.91,2459.80,4940428,-1.6079
1500,2022-01-03,TCS,3812.18,3897.68,3782.48,3840.08,3823078,1.0547
3000,2022-01-03,WIPRO,418.23,421.81,409.35,415.58,2384649,-1.0526
750,2022-01-03,INFOSYS,1397.96,1408.30,1366.67,1387.49,3163512,-0.8938
2250,2022-01-03,HDFC,1659.99,1700.76,1650.49,1675.63,2309136,1.5531


In [8]:

WINDOW_SIZE = 10

def create_sequences(data, window):
    X, y = [], []
    for i in range(len(data) - window):
        X.append(data[i:i+window])
        y.append(data[i+window])
    return np.array(X), np.array(y)

close_prices = df['close'].values.reshape(-1,1)

scaler = MinMaxScaler()
scaled = scaler.fit_transform(close_prices)

X, y = create_sequences(scaled, WINDOW_SIZE)

split = int(len(X)*0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]


In [9]:

model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(WINDOW_SIZE,1)),
    LSTM(50),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')
model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5
94/94 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - loss: 0.0736 - val_loss: 0.0789
Epoch 2/5
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0651 - val_loss: 0.0760
Epoch 3/5
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0630 - val_loss: 0.0731
Epoch 4/5
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0618 - val_loss: 0.0738
Epoch 5/5
94/94 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 0.0574 - val_loss: 0.0674


In [10]:

pred = model.predict(X_test)

pred = scaler.inverse_transform(pred)
y_test_actual = scaler.inverse_transform(y_test)

rmse = np.sqrt(mean_squared_error(y_test_actual, pred))
print("RMSE:", rmse)


24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step
RMSE: 1001.6683953825142


In [11]:

def moving_avg(data, window):
    preds = []
    for i in range(len(data)-window):
        preds.append(np.mean(data[i:i+window]))
    return np.array(preds)

baseline_pred = moving_avg(close_prices.flatten(), WINDOW_SIZE)
baseline_pred = baseline_pred[-len(y_test_actual):]

baseline_rmse = np.sqrt(mean_squared_error(y_test_actual.flatten(), baseline_pred))
print("Baseline RMSE:", baseline_rmse)


Baseline RMSE: 1122.6661933749383



## AI Usage

Prompt:
Create LSTM model for stock prediction

Critique:
Modified sequence logic, added time-series split, added baseline comparison
